# 2026-11-02 小 CNN 训练闭环（重建版）

> 原笔记本因 11/2 晚助手目录整理误删；本文件按 11/2 讲义代码重建，代码与已跑通的流程一致。
> **真实结果已保存在** `learning-lab\results\day1102\20261102_170336\`（config.json / history.json / model_weights.pt / training_curves.png），无需重新训练即可核验。
> 真实结果：train 79.74% / val 76.60%（第5轮），最佳验证在第4轮 val 78.40%。


In [ ]:
import json, time
from pathlib import Path
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

torch.manual_seed(42)

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 7 * 7, 10),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

data_root = Path(r"E:\develop\Graduate_AI\learning-lab\data")
tf = transforms.ToTensor()
train_full = datasets.FashionMNIST(data_root, train=True, download=False, transform=tf)
test_full  = datasets.FashionMNIST(data_root, train=False, download=False, transform=tf)

split_path = Path(r"E:\develop\Graduate_AI\learning-lab\results\day22\split_indices.json")
split = json.loads(split_path.read_text(encoding="utf-8"))
train_idx = split["train_indices"][:5000]
val_idx   = split["val_indices"][:1000]

train_set = Subset(train_full, train_idx)
val_set   = Subset(train_full, val_idx)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_set, batch_size=64, shuffle=False, num_workers=0)

xb, yb = next(iter(train_loader))
print("一个 batch：", xb.shape, yb.shape, xb.dtype)
print("SmallCNN 参数：", sum(p.numel() for p in SmallCNN().parameters()))


In [ ]:
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            z = model(xb)
            total_loss += loss_fn(z, yb).item() * yb.size(0)
            correct += (z.argmax(dim=1) == yb).sum().item()
            n += yb.size(0)
    return total_loss / n, correct / n

def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for xb, yb in loader:
        z = model(xb)
        loss = loss_fn(z, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * yb.size(0)
        correct += (z.argmax(dim=1) == yb).sum().item()
        n += yb.size(0)
    return total_loss / n, correct / n


In [ ]:
# 第 1 步：8 张过拟合 sanity check
loss_fn = nn.CrossEntropyLoss()
model_sanity = SmallCNN()
opt = torch.optim.SGD(model_sanity.parameters(), lr=0.05)
xb8, yb8 = next(iter(DataLoader(Subset(train_full, train_idx[:8]), batch_size=8)))
for it in range(200):
    z = model_sanity(xb8)
    loss = loss_fn(z, yb8)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if it % 40 == 0 or it == 199:
        acc = (z.argmax(dim=1) == yb8).float().mean().item()
        print(f"iter {it:3d}  loss={loss.item():.4f}  acc={acc:.2f}")


In [ ]:
# 第 2 步：1 个 epoch，先计时
model = SmallCNN()
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.SGD(model.parameters(), lr=0.1)
t0 = time.time()
tr_loss, tr_acc = train_one_epoch(model, train_loader, loss_fn, opt)
t1 = time.time()
va_loss, va_acc = evaluate(model, val_loader, loss_fn)
print(f"1 epoch 用时 {t1-t0:.1f}s | train loss {tr_loss:.4f} acc {tr_acc:.4f} | val loss {va_loss:.4f} acc {va_acc:.4f}")


In [ ]:
# 第 3 步：5 轮完整训练＋history＋曲线＋保存（主产出）
EPOCHS = 5
model = SmallCNN()
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.SGD(model.parameters(), lr=0.1)

run_id  = time.strftime("%Y%m%d_%H%M%S")
run_dir = Path(r"E:\develop\Graduate_AI\learning-lab\results\day1102") / run_id
run_dir.mkdir(parents=True, exist_ok=True)

history, total_steps, t_start = [], 0, time.time()
for ep in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, loss_fn, opt)
    va_loss, va_acc = evaluate(model, val_loader, loss_fn)
    steps = len(train_loader)
    total_steps += steps
    history.append({"epoch": ep, "train_loss": tr_loss, "val_loss": va_loss,
                    "train_accuracy": tr_acc, "val_accuracy": va_acc,
                    "epoch_steps": steps, "total_steps": total_steps})
    print(f"ep{ep} train {tr_loss:.4f}/{tr_acc:.4f}  val {va_loss:.4f}/{va_acc:.4f}")

elapsed = time.time() - t_start
print(f"总用时 {elapsed:.1f}s（{elapsed/EPOCHS:.1f}s/轮），总更新 {total_steps} 次")

config = {
    "dataset": "FashionMNIST",
    "data_policy": "split_indices.json: train_indices[:5000] / val_indices[:1000]",
    "train_n": len(train_set), "val_n": len(val_set),
    "model": "SmallCNN(Conv1-8-3-P1, Conv8-16-3-P1, MaxPool2 x2, FC 784-10)",
    "params": sum(p.numel() for p in model.parameters()),
    "seed": 42, "batch_size": 64, "learning_rate": 0.1,
    "optimizer": "SGD", "loss": "CrossEntropyLoss",
    "epochs": EPOCHS, "steps_per_epoch": len(train_loader),
    "shuffle": True, "preprocess": "ToTensor only（无 Normalize）",
    "device": "cpu", "torch_version": torch.__version__,
    "elapsed_sec": round(elapsed, 1),
    "metric_timing": "每轮训练结束后，用同一模型完整评价训练/验证子集",
}
(run_dir / "config.json").write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")
(run_dir / "history.json").write_text(json.dumps(history, ensure_ascii=False, indent=2), encoding="utf-8")
torch.save(model.state_dict(), run_dir / "model_weights.pt")
print("保存到：", run_dir)


In [ ]:
# 画训练/验证曲线并存 png
eps = [h["epoch"] for h in history]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(eps, [h["train_loss"] for h in history], "o-", label="train")
axes[0].plot(eps, [h["val_loss"] for h in history], "s--", label="val")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss"); axes[0].legend(); axes[0].set_title("loss")
axes[1].plot(eps, [h["train_accuracy"] for h in history], "o-", label="train")
axes[1].plot(eps, [h["val_accuracy"] for h in history], "s--", label="val")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("accuracy"); axes[1].legend(); axes[1].set_title("accuracy")
fig.tight_layout()
fig.savefig(run_dir / "training_curves.png", dpi=120)
plt.show()


## 真实结果（已保存在 learning-lab，无需重训核验）

`learning-lab\results\day1102\20261102_170336\history.json`：

| epoch | train_loss | train_acc | val_loss | val_acc |
|---|---|---|---|---|
| 1 | 1.4722 | 50.08% | 1.0639 | 62.40% |
| 2 | 0.8785 | 69.84% | 1.0081 | 67.30% |
| 3 | 0.7145 | 74.48% | 1.1161 | 71.00% |
| 4 | 0.6199 | 78.34% | 0.6296 | **78.40%** |
| 5 | 0.5628 | **79.74%** | 0.6855 | 76.60% |

- 最佳验证点：第 4 轮 val_acc 78.40%、val_loss 0.6296
- 第 5 轮验证 loss 回升（0.6296→0.6855），属 5 轮内轻微过拟合/抖动
- 与线性基线(7850 参数, val 80.00%)对比：CNN(9098) 76.60% < 线性 80.00%，照实记录


In [ ]:
# 受控变式 a：padding=0 → 前向报错
m = SmallCNN()
m.features[0] = nn.Conv2d(1, 8, kernel_size=3, padding=0)
m.features[3] = nn.Conv2d(8, 16, kernel_size=3, padding=0)
with torch.no_grad():
    try:
        m(torch.zeros(1, 1, 28, 28))
    except RuntimeError as e:
        print("报错：", e)


In [ ]:
# 受控变式 b：MaxPool vs 2x2 步长卷积
x = torch.zeros(1, 8, 28, 28)
mp = nn.MaxPool2d(2)
cl = nn.Conv2d(8, 8, kernel_size=2, stride=2, bias=False)
with torch.no_grad():
    print("MaxPool 输出：", mp(x).shape, "参数量：", sum(p.numel() for p in mp.parameters()))
    print("步长卷积输出：", cl(x).shape, "参数量：", sum(p.numel() for p in cl.parameters()))


In [ ]:
# 变式：lr 0.1 vs 0.01 各跑 1 轮
loss_fn = nn.CrossEntropyLoss()
for lr in (0.1, 0.01):
    torch.manual_seed(42)
    m = SmallCNN()
    o = torch.optim.SGD(m.parameters(), lr=lr)
    l, a = train_one_epoch(m, train_loader, loss_fn, o)
    vl, va = evaluate(m, val_loader, loss_fn)
    print(f"lr={lr}: train {l:.4f}/{a:.4f}  val {vl:.4f}/{va:.4f}")
